In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'orjson', 'polars', 'pyarrow'])
import os, gc, gzip, orjson, random, csv
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
RAW_REVIEW_PATH = '/kaggle/input/datasets/b22dckh072/file01/Clothing_Shoes_and_Jewelry.jsonl/Clothing_Shoes_and_Jewelry.jsonl'
RAW_META_PATH   = '/kaggle/input/datasets/b22dckh072/file01/meta_Clothing_Shoes_and_Jewelry.jsonl/meta_Clothing_Shoes_and_Jewelry.jsonl'

WORKING_DIR = '/kaggle/working/processed'
os.makedirs(WORKING_DIR, exist_ok=True)

INTERIM_RAW_PATH = '/kaggle/working/interim_raw.parquet'
TRAIN_OUT_PATH = os.path.join(WORKING_DIR, 'train_interactions.parquet')
TEST_OUT_PATH  = os.path.join(WORKING_DIR, 'test_interactions.parquet')
META_OUT_PATH  = os.path.join(WORKING_DIR, 'filtered_metadata.parquet')
MAPPING_OUT_PATH = os.path.join(WORKING_DIR, 'item_mapping.parquet')

K_CORE = 5
TRAIN_RATIO = 0.8
CHUNK_SIZE = 3000000 

In [3]:
def extract_valid_fashion_items(meta_path):
    print("BƯỚC 0: Quét trước Meta Data để tìm các sản phẩm AMAZON FASHION hợp lệ...")
    valid_fashion_asins = set()
    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                # Dùng exact match để lọc đúng "AMAZON FASHION"
                if data.get('main_category') == "AMAZON FASHION":
                    asin = data.get('parent_asin')
                    if asin:
                        valid_fashion_asins.add(asin)
            except Exception:
                continue
    print(f"-> Tìm thấy {len(valid_fashion_asins):,} sản phẩm thuộc AMAZON FASHION.")
    return valid_fashion_asins

In [4]:
def jsonl_to_parquet_filtered(input_path, output_path, chunk_size, valid_fashion_asins):
    print("BƯỚC 1: Chuyển đổi file Review 27GB sang Parquet (Chỉ lấy Fashion và thêm thuộc tính)...")
    writer = None
    buffer = []
    chunk_count = 0

    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')
                
                # LỌC NGAY TỪ ĐẦU: Chỉ xử lý review nếu sản phẩm là thời trang
                if asin in valid_fashion_asins:
                    buffer.append({
                        'user_id': data.get('user_id'),
                        'parent_asin': asin,
                        'rating': data.get('rating'),
                        'timestamp': data.get('timestamp'),
                        'helpful_vote': data.get('helpful_vote', 0), 
                        'verified_purchase': data.get('verified_purchase', False)
                    })
                
                # Khi buffer đầy, xả xuống ổ cứng
                if len(buffer) >= chunk_size:
                    chunk_count += 1
                    table = pa.Table.from_pandas(pd.DataFrame(buffer))
                    if writer is None:
                        writer = pq.ParquetWriter(output_path, table.schema)
                    writer.write_table(table)
                    buffer.clear()
                    print(f"   Đã ghi xong Chunk thứ {chunk_count}")
                    gc.collect()
            except Exception:
                continue

    # Ghi phần dư
    if buffer:
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)

    if writer: writer.close()
    print("-> Hoàn tất nạp dữ liệu thô đã được thanh lọc!")

In [5]:
def apply_k_core(data_path, k=5):
    print(f"BƯỚC 2: Áp dụng K-Core={k} (Tự động lặp cho đến khi hội tụ)...")
    lf = pl.scan_parquet(data_path).select(['user_id', 'parent_asin'])
    curr_data = lf.collect()
    
    iteration = 1
    while True:
        print(f"   Vòng lặp K-Core thứ {iteration}...")
        
        # Ghi nhận số lượng dòng hiện tại trước khi lọc
        prev_height = curr_data.height
        
        # 1. Lọc User có ít nhất K tương tác
        user_counts = curr_data.group_by('user_id').len()
        valid_users = user_counts.filter(pl.col('len') >= k).select('user_id')
        curr_data = curr_data.join(valid_users, on='user_id', how='inner')

        # 2. Lọc Item có ít nhất K tương tác
        item_counts = curr_data.group_by('parent_asin').len()
        valid_items = item_counts.filter(pl.col('len') >= k).select('parent_asin')
        curr_data = curr_data.join(valid_items, on='parent_asin', how='inner')

        del user_counts, valid_users, item_counts, valid_items
        gc.collect()
        
        # Lấy các dòng duy nhất
        curr_data = curr_data.unique()
        
        # Lấy số lượng dòng sau khi lọc
        curr_height = curr_data.height
        
        print(f"      -> Số tương tác còn lại: {curr_height:,} (Đã xóa {prev_height - curr_height:,} dòng)")
        
        # ĐIỀU KIỆN HỘI TỤ: Nếu số dòng không giảm đi nữa thì dừng lại
        if curr_height == prev_height:
            print(f"   => K-Core ĐÃ HỘI TỤ SAU {iteration} VÒNG LẶP!")
            break
            
        iteration += 1

    return curr_data

In [6]:
def split_and_map(raw_path, valid_ids, train_out, test_out, ratio):
    print("BƯỚC 3: Map ID và Phân tách Train/Test theo MỐC THỜI GIAN CHUNG...")

    valid_user_set = set(valid_ids['user_id'].to_list())
    valid_item_set = set(valid_ids['parent_asin'].to_list())

    u_map = pl.DataFrame({'user_id': list(valid_user_set)}).with_row_index('mapped_user_id', offset=1)
    i_map = pl.DataFrame({'parent_asin': list(valid_item_set)}).with_row_index('mapped_item_id', offset=1)
    
    u_map = u_map.with_columns(pl.col('mapped_user_id').cast(pl.Int32))
    i_map = i_map.with_columns(pl.col('mapped_item_id').cast(pl.Int32))

    # Nạp dữ liệu và gắn ID. Bây giờ lấy thêm cả 2 cột mới
    df_clean = (pl.scan_parquet(raw_path)
                .join(u_map.lazy(), on='user_id', how='inner')
                .join(i_map.lazy(), on='parent_asin', how='inner')
                .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp', 'helpful_vote', 'verified_purchase'])
                .collect())

    df_clean = df_clean.sort('timestamp')
    
    cutoff_idx = int(df_clean.height * ratio)
    
    cutoff_timestamp = df_clean['timestamp'][cutoff_idx]
    print(f"-> Mốc thời gian chung để cắt hệ thống là: {cutoff_timestamp}")

    train_data = df_clean.filter(pl.col('timestamp') <= cutoff_timestamp)
    test_data  = df_clean.filter(pl.col('timestamp') > cutoff_timestamp)

    print("-> Đang loại bỏ các User/Item mới xuất hiện trong Test mà Train chưa có...")
    train_users = train_data.select('mapped_user_id').unique()
    train_items = train_data.select('mapped_item_id').unique()
    
    test_data = (test_data
                 .join(train_users, on='mapped_user_id', how='inner')
                 .join(train_items, on='mapped_item_id', how='inner'))

    # Ghi xuống đĩa cứng
    train_data.write_parquet(train_out)
    test_data.write_parquet(test_out)

    print(f"-> Train: {train_data.height:,} dòng | Test: {test_data.height:,} dòng.")
    return valid_item_set, i_map

In [7]:
def process_meta(meta_path, valid_items_set, item_map_df, output_path, chunk_size):
    import re
    print("BƯỚC 4: Lọc và Xử lý Meta Data 17GB...")
    writer = None
    buffer = []
    
    def clean_number(val):
        try:
            if isinstance(val, str):
                nums = re.findall(r'\d+\.?\d*', val)
                return float(nums[0]) if nums else 0.0
            return float(val) if val is not None else 0.0
        except:
            return 0.0

    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')

                if asin in valid_items_set:
                    buffer.append({
                        'parent_asin': asin,
                        'price': clean_number(data.get('price')),
                        'average_rating': clean_number(data.get('average_rating')),
                        'rating_number': int(clean_number(data.get('rating_number'))),
                        'store': str(data.get('store', 'Unknown')),
                        'categories': str(data.get('categories', '[]'))
                    })

                if len(buffer) >= chunk_size:
                    df_chunk = pd.DataFrame(buffer)
                    table = pa.Table.from_pandas(df_chunk)
                    if writer is None:
                        writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
                    writer.write_table(table)
                    buffer.clear()
                    gc.collect()

            except Exception:
                continue

    if buffer:
        df_chunk = pd.DataFrame(buffer)
        table = pa.Table.from_pandas(df_chunk)
        if writer is None:
            writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
        writer.write_table(table)

    if writer: writer.close()

    print("   Đang gắn Map ID cho Meta Data...")
    (pl.scan_parquet(output_path + ".tmp")
     .join(item_map_df.lazy(), on='parent_asin', how='inner')
     .drop('parent_asin') 
     .sink_parquet(output_path))

    os.remove(output_path + ".tmp")
    print("-> Hoàn tất xử lý Meta Data!")

In [8]:
valid_fashion_asins = extract_valid_fashion_items(RAW_META_PATH)
jsonl_to_parquet_filtered(RAW_REVIEW_PATH, INTERIM_RAW_PATH, CHUNK_SIZE, valid_fashion_asins)
valid_ids_df = apply_k_core(INTERIM_RAW_PATH, K_CORE)
valid_items, item_mapping = split_and_map(INTERIM_RAW_PATH, valid_ids_df, TRAIN_OUT_PATH, TEST_OUT_PATH, TRAIN_RATIO)
item_mapping.write_parquet(MAPPING_OUT_PATH)
print(f"-> Đã lưu file Mapping tại: {MAPPING_OUT_PATH}")
process_meta(RAW_META_PATH, valid_items, item_mapping, META_OUT_PATH, CHUNK_SIZE)

# Dọn dẹp rác tạm thời
if os.path.exists(INTERIM_RAW_PATH):
    os.remove(INTERIM_RAW_PATH)

BƯỚC 0: Quét trước Meta Data để tìm các sản phẩm AMAZON FASHION hợp lệ...
-> Tìm thấy 6,038,522 sản phẩm thuộc AMAZON FASHION.
BƯỚC 1: Chuyển đổi file Review 27GB sang Parquet (Chỉ lấy Fashion và thêm thuộc tính)...
   Đã ghi xong Chunk thứ 1
   Đã ghi xong Chunk thứ 2
   Đã ghi xong Chunk thứ 3
   Đã ghi xong Chunk thứ 4
   Đã ghi xong Chunk thứ 5
   Đã ghi xong Chunk thứ 6
   Đã ghi xong Chunk thứ 7
   Đã ghi xong Chunk thứ 8
   Đã ghi xong Chunk thứ 9
   Đã ghi xong Chunk thứ 10
   Đã ghi xong Chunk thứ 11
   Đã ghi xong Chunk thứ 12
   Đã ghi xong Chunk thứ 13
   Đã ghi xong Chunk thứ 14
   Đã ghi xong Chunk thứ 15
   Đã ghi xong Chunk thứ 16
   Đã ghi xong Chunk thứ 17
   Đã ghi xong Chunk thứ 18
   Đã ghi xong Chunk thứ 19
   Đã ghi xong Chunk thứ 20
-> Hoàn tất nạp dữ liệu thô đã được thanh lọc!
BƯỚC 2: Áp dụng K-Core=5 (Tự động lặp cho đến khi hội tụ)...
   Vòng lặp K-Core thứ 1...
      -> Số tương tác còn lại: 23,494,150 (Đã xóa 36,599,284 dòng)
   Vòng lặp K-Core thứ 2...
  